# 00 — Data, universe and conventions

The universe, dates and conventions every later notebook imports from `maplab.contract`; how the `Panel` keeps every estimate walk-forward; and the correlation facts behind the UUP-heavy weights of the risk-based methods (those that need no forecast of μ). Nothing here allocates, and every statistic is descriptive.

## Setup

In [ ]:
# Colab only: get the repo and install it. Does nothing when run locally.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    repo = "/content/multi-asset-portfolio-lab"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/FranQuant/multi-asset-portfolio-lab", repo], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo], check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import maplab as ml
from helpers import nb00 as h
from IPython.display import display

from maplab import Panel, apply_style

In [ ]:
apply_style()
prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)    # for estimation (mu, Sigma)
rf = ml.load_rf_returns()
panel = Panel({"prices": prices, "returns": log_returns, "rf": rf})
assert len(prices) == 4611 and prices.index[-1] == pd.Timestamp("2026-04-30"), "data window changed: update the §2 text"

## §1 Universe and conventions

13 ETFs, one per risk premium, in 6 groups; UUP is a fund long the US dollar against a basket of major currencies. BIL, a T-bill ETF, is the cash proxy and serves as the risk-free rate; it is never allocated. Every notebook imports these constants from `maplab.contract` rather than redefining them.

In [ ]:
groups = pd.Series(
    {g: ", ".join(t) for g, t in ml.ASSET_GROUPS.items()}
).rename("tickers").to_frame()
groups["n"] = [len(v) for v in ml.ASSET_GROUPS.values()]
ml.reporting.show_table(groups[["n", "tickers"]], int_cols=["n"])

Methods in this repo fall into four groups. Return-based methods need a forecast of μ: MaxSharpe (notebook 01) and Black–Litterman (03). Risk-based methods need no μ forecast: GMV (01), BetaTargetMinVar (02, Σ plus a β floor), MDP (05), ERC (06) and HRP (07). EW and 60/40 are the benchmarks, and notebook 10 adds a volatility overlay on top of the others. Notebook 04 swaps the covariance estimator under GMV, MaxSharpe and BL.

Fixed once in `maplab.contract` and the harness (`src/maplab/`); the main constants are printed below.

- Returns: log returns for estimation, simple returns for compounding.
- Risk-free: BIL daily simple returns; Sharpe $= \mathrm{mean}(r - r_f)\,/\,\mathrm{sd}(r - r_f)\cdot\sqrt{252}$.
- Rebalancing: at each calendar month-end label, weights are re-estimated on the trailing 252 trading days (walk-forward, §2). The first label with a full window is 2009-01-31: 208 rebalances, 168 up to the 2022-12-31 split and 40 after it.
- Turnover: one-way, measured from the drifted weights (the previous weights after a month of price moves) to the new targets, annualized by the calendar. Costs: 10 bps one-way, charged on the first trading day the new weights are held.
- Constraints: an attribute of each method, not a global rule.
- Data: forward-fill only, never back-fill; all 13 assets trade from 2008-01-02.
- Benchmarks: equal weight (EW, 1/13 in each asset) and 60/40 (60% SPY, 40% IEF), rebalanced monthly like every other run; neither is registered.

In [ ]:
rf_full_ann = float(rf["BIL"].mean() * ml.TRADING_DAYS)

print("Window      :", ml.START, "→", ml.END)
print("Rebalance   :", ml.REBALANCE, "(month-end)")
print("Cov lookback:", ml.COV_LOOKBACK, "days")
print("Split       :", ml.TRAIN_TEST_SPLIT)
print("Risk-free   :", ml.RF_TICKER, f"({rf_full_ann:.2%} ann., full-window)")
print("Cost        :", f"{ml.COST_BPS:.0f} bps one-way")

## §2 Data and the walk-forward Panel

Prices come from one cached file, `data/cache/prices.parquet`, which `scripts/build_panel.py` builds on SPY's trading calendar with forward-fill only, from `data/raw/eod_prices.csv`, a CSV with columns Date, Ticker, AdjClose (the panel built from the vendor archive ships as `data/cache/prices.parquet`). `ml.load_prices()` returns the 13 allocatable assets; BIL is read separately by `ml.load_rf_returns()`. The window opens 2008-01-01; the first price is 2008-01-02 and the last 2026-04-30 (4,611 trading days), and the first return 2008-01-03.

Every backtest in this repo is walk-forward: at each rebalance label $t$, a method sees only `panel.slice(t, …)`, the returns dated strictly before $t$, and holds its weights until the next label. The cell below checks this at 2020-03-31: the last row used is 2020-03-30.

In [ ]:
# demonstrate the gatekeeper
asof = pd.Timestamp("2020-03-31")
window = panel.slice(asof, "returns", lookback=ml.COV_LOOKBACK)
print(f"slice(asof={asof.date()}, lookback=252):")
print(f"  last available date: {window.index[-1].date()}")
assert window.index.max() < asof, "no-look-ahead violated!"
print("  ✓ all data strictly before asof")

## §3 Correlation statistics

At each rebalance label $t$ the statistics use the same window as the methods: the 252 daily log returns dated strictly before $t$, with Pearson correlations $\rho_{ij}(t)$. With $U$ = UUP and $R$ the other 12 assets:

- UUP's average correlation with the rest: $\bar\rho_U(t) = \frac{1}{12}\sum_{j \in R} \rho_{Uj}(t)$
- UUP's annualized volatility: $\sigma_U(t) = \sqrt{252}\;\mathrm{sd}_t(r_U)$
- the average pairwise correlation among the other 12: $\bar\rho_{\mathrm{rest}}(t) = \frac{1}{12 \cdot 11}\sum_{i \in R}\,\sum_{j \in R,\, j \neq i} \rho_{ij}(t)$

Notebooks 01, 05 and 07 show how GMV, MDP and HRP weights respond to these quantities. §4 computes the full-sample versions over every return from 2008-01-03 to 2026-04-30.

## §4 Full-sample structure

Annualized volatility by asset, then the full-sample correlation matrix.

In [ ]:
assert not prices.isna().any().any(), "missing prices after forward-fill"
ann_vol = log_returns[ml.UNIVERSE].std() * np.sqrt(ml.TRADING_DAYS)
vol_table = pd.DataFrame({
    "group": [ml.GROUP_OF[t] for t in ml.UNIVERSE],
    "ann_vol_%": 100 * ann_vol[ml.UNIVERSE],
    "rank (1 = lowest)": ann_vol[ml.UNIVERSE].rank().astype(int),
}).rename_axis(None)
ml.reporting.show_table(vol_table, dp=1, int_cols=["rank (1 = lowest)"])

Annualized volatility runs from 6.3% (TIP) to 29.8% (VNQ). UUP, at 8.1%, is third-lowest after TIP and IEF, so its volatility is not what sets it apart; its correlations are.

The full-sample correlation matrix, ordered by hierarchical clustering (the ordering HRP uses, notebook 07), values annotated, UUP's row and column boxed.

In [ ]:
fig, _, _ = h.clustered_corr_figure(log_returns)
plt.show()

C_full = log_returns[ml.UNIVERSE].corr()
uup_corr = C_full["UUP"].drop("UUP").sort_values().rename_axis(None)
print("UUP full-sample correlations with the other 12:")
display(ml.reporting.show_table(uup_corr.to_frame("ρ with UUP"), dp=2))
neg_all = [a for a in ml.UNIVERSE if (C_full[a].drop(a) < 0).all()]
print("assets negatively correlated with all 12 others:", neg_all)

All 12 of UUP's full-sample correlations are negative, from −0.07 (TLT) to −0.42 (GLD); it is the only asset negatively correlated with all 12 others.

## §5 UUP and the rest through time

The §3 statistics at the 208 rebalance labels, then their medians over the train (168 labels, to 2022-12-31) and test (40 labels) windows and their 2022-12-31 values.

In [ ]:
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

fig, uup_table = h.uup_rolling_figure(panel, rdates, split_ts)
plt.show()

uup_summary = pd.DataFrame({
    "train (<= split)": uup_table.loc[uup_table.index <= split_ts].median(),
    "test (> split)": uup_table.loc[uup_table.index > split_ts].median(),
    "2022-12-31": uup_table.loc[pd.Timestamp("2022-12-31")],
}).T
uup_summary = uup_summary.rename(columns={"uup_mean_corr": "ρ̄_U", "sigma_UUP": "σ_U (%)", "rho_bar_rest": "ρ̄_rest"})
uup_summary["σ_U (%)"] *= 100
ml.reporting.show_table(uup_summary, dp=2, col_dp={"σ_U (%)": 1})

After 2022 the median ρ̄_U falls from −0.26 to −0.45 and the median ρ̄_rest rises from 0.24 to 0.44, while σ_U barely moves (7.5% → 7.4%). UUP became more of a hedge against the other 12 just as they started moving together. The two steps in σ_U, up at 2020-03-31 and down at 2021-03-31, are March 2020 entering and then leaving the 252-day window.

## §6 TAKEAWAY — UUP is negatively correlated with every other asset, and the other 12 became more correlated after 2022

- **One contract.** 13 ETFs in 6 groups, BIL as the risk-free rate, month-end rebalancing on a trailing 252-day window, 10 bps one-way, split at 2022-12-31: 208 rebalances, 40 in the test window. EW and 60/40 are the benchmarks; neither is registered.
- **Walk-forward by construction.** `Panel.slice` returns only rows dated strictly before the label; estimation uses log returns, compounding simple returns.
- **UUP is the exception in the full sample.** Its 12 correlations run from −0.07 to −0.42; no other asset is negatively correlated with every other.
- **After 2022 two things move.** Medians: ρ̄_U −0.26 → −0.45 and ρ̄_rest 0.24 → 0.44, while σ_U barely moves (7.5% → 7.4%).
- **What the data cannot tell us.** §4 is full-sample hindsight, which no allocation uses. The test window is 40 months and was not a sealed holdout: the universe and Black–Litterman's k were set while the full sample was available (k after a 2022-12-31 snapshot).

The UUP-heavy weights of the risk-based methods start from one asset negatively correlated with all the others, in a book whose other 12 assets moved together more after 2022.